In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from umap import UMAP
from pathlib import Path
from langchain_core.documents import Document

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지


DOCS_PATH = "../data/RAG/maple_items_documents.json"

In [ ]:
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

```plain_text
A가 전달한 all_documents / all_chunks
                ↓
        page_content 추출
                ↓
        Embedding 모델 준비
                ↓
          문서 임베딩
                ↓
            embeddings
                ↓
       다음 Vector DB 단계로 전달
```

| 구분          | 내용                                                   |
| ----------- | ---------------------------------------------------- |
| 입력          | `all_documents` 또는 `all_chunks`                      |
| 입력 형태       | `list[Document]`                                     |
| 핵심 작업       | Document의 `page_content`를 임베딩 벡터로 변환                 |
| 출력          | `embeddings`                                         |
| 출력 형태       | 문서마다 하나의 숫자 벡터                                       |
| 다음 담당자에게 전달 | `all_documents` + `embeddings` + 사용한 embedding model |


### document 타입 확인

In [ ]:
data = "../data/RAG" + "document_json_파일"

print(type(data))
print(type(data[0]))


### 가져온 문서가 document 타입이 아닐 경우 변환하는 코드

In [ ]:
def load_documents(file_path):
    with open(file_path, "r", encoding='utf-8') as f:
        data = json.load(f)

    documents = [
        Document(page_content=doc['page_content'], metadata=doc['metadata']) for doc in data
    ]
    return documents

guide_documents = load_documents("maple_guides_documents.json")
jobs_documents = load_documents("maple_jobs_documents.json")
items_documents = load_documents("maple_items_documents.json")


### 문서 통합, 임베딩

In [ ]:
all_documents = (guide_documents + items_documents + jobs_documents)

embeddings = model.encode(all_documents)

print("임베딩 배열 모양:", embeddings.shape)
display(embeddings)